**TRABAJO DE CLASIFICACION - GRUPO 04**

### Importación de librerias

In [1]:
#Importamos librerias necesarias para exploración de datos
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import math as mt
import sklearn.model_selection as model_selection
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.compose import make_column_selector
from sklearn.naive_bayes import ComplementNB
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.base import TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from imblearn.combine import SMOTETomek
import joblib

### Importación de Dataset

In [2]:
#Establecemos la ruta de la carpeta de trabajo y seleccionamos el archivo del dataset
path_base = r"C:\Users\ABUDABI\Downloads\CURSOS_ENEI_2026\ML_PRODUCCION_WEB\TRABAJO FINAL\CLASIFICACION"
dataset = pd.read_csv(f'{path_base}/datos_cloro.csv', sep = ',')

#Inspeccionamos los 5 primeros registros del dataset
dataset.head(5)

,ID,CPCSD,Recibe_Cuota_Familiar_SI,Flag_Menos2anios_Eleccion,tiene_mujer_lider,Flag_Miembros_SI_SecundariaSuperior,Operarios_Con_Incentivos_Pagos,Operarios_Gasfiteros,Cuenta_ConFondos_SI,Registro_Cloro_Residual_SI,...,Flag_Capa_Gestion,Flag_Capa_Operacion,Tipo_de_Fuente_Superficial,Costos_ServSaneamiento_XCuotaFamiliar_SI,Documentos_Tecnicos,Apoyo_Tecnico,Apoyo_Infraestructura,Documentos_Financieros,CapaOperacion_x_Gasfitero,y
0,1,Compacta,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Compacta,0,1,0,1,0,1,0,0,...,0,0,0,0,0,0,0,2,0,1
2,3,Compacta,0,1,1,0,0,1,0,1,...,1,1,0,0,3,3,1,3,1,1
3,4,Vacio,1,0,0,1,0,0,0,0,...,1,1,0,0,0,2,0,2,0,0
4,5,Compacta,1,0,1,1,0,0,0,0,...,0,1,0,1,0,0,0,2,0,0


### Determinación X e Y

In [3]:
features_modelo = [
    'Flag_Menos2anios_Eleccion',
    'Establecimiento_Vigila_Calidad_Agua_SI',
    'ApoyoMuni_Control_Calidad_Agua_SI',
    'cuotapromedio_Cat',
    'Flag_Miembros_SI_SecundariaSuperior',
    'Registro_Cloro_Residual_SI',
    'Realizan_Limpieza_SitemaAgua_SI',
    'Apoyo_Infraestructura',
    'Salud_Financiera_Cuotas_Porc',
    'Apoyo_Tecnico'
]

X = dataset[features_modelo]
y = dataset['y']

# Primera división: 85% para entrenamiento+validación, 15% para prueba final
X_temp, X_final, y_temp, y_final = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=123,
    stratify=y,
    shuffle=True
)

# Segunda división: 70% entrenamiento, 15% validación (del 85% restante)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,  # 15% del total
    random_state=123,
    stratify=y_temp,
    shuffle=True
)

print(f"\nDivisión completada:")
print(f"   Entrenamiento (70%): {len(X_train):,} registros")
print(f"   Validación (15%): {len(X_val):,} registros")
print(f"   Prueba Final (15%): {len(X_final):,} registros")


División completada:
   Entrenamiento (70%): 13,205 registros
   Validación (15%): 2,821 registros
   Prueba Final (15%): 2,829 registros


### Preprocesamiento

In [4]:
# Identificar columnas numéricas (todas tus variables lo son)
categorical_columns = X_train.select_dtypes(include='object').columns.tolist()
numerical_columns = X_train.select_dtypes(include='number').columns.tolist()

print(f"\nVariables categóricas ({len(categorical_columns)}):")
for col in categorical_columns[:5]:
    print(f"  - {col}")

print(f"\nVariables numéricas ({len(numerical_columns)}):")
for col in numerical_columns[:5]:
    print(f"  - {col}")

# Crear preprocesador
prep = ColumnTransformer([
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns),
    ('scaler', StandardScaler(), numerical_columns)
], remainder='drop')


Variables categóricas (1):
  - cuotapromedio_Cat

Variables numéricas (9):
  - Flag_Menos2anios_Eleccion
  - Establecimiento_Vigila_Calidad_Agua_SI
  - ApoyoMuni_Control_Calidad_Agua_SI
  - Flag_Miembros_SI_SecundariaSuperior
  - Registro_Cloro_Residual_SI


### Configuración de validación cruzada

In [5]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=123)
print(f"Validación cruzada: 5 folds, 2 repeticiones")

Validación cruzada: 5 folds, 2 repeticiones


### Optimización con Optuna

In [6]:
# Scikit-learn
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                           classification_report, roc_curve, auc, f1_score,
                           precision_score, recall_score)
from sklearn.inspection import permutation_importance

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Imbalanced-learn
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as Pipeline_imb
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Optuna
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

In [8]:
def optimize_model(model_name, X_train, y_train, cv, n_trials=30):

    #Función para optimizar hiperparámetros con manejo de errores

    print(f"Optimizando: {model_name}")

    def create_pipeline(trial):

        # Estrategia de balanceo adaptativa
        sampling_method = trial.suggest_categorical("sampling_method",["none", "smote", "smotetomek"])
        sampling_strategy = trial.suggest_float("sampling_strategy",0.6, 0.8)

        if sampling_method == "none":
            sampler = 'passthrough'
        elif sampling_method == "smote":
            sampler = SMOTE(sampling_strategy=sampling_strategy,random_state=123)
        else:
            try:
                sampler = SMOTETomek(
                    sampling_strategy=sampling_strategy,
                    random_state=123
                )
            except:
                sampler = SMOTE(
                    sampling_strategy=sampling_strategy,
                    random_state=123
                )

        # Configurar parámetros según el modelo
        if model_name == "XGBoost":
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 8),
                'n_estimators': trial.suggest_int('n_estimators', 80, 250, step=40),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 6),
                'gamma': trial.suggest_float('gamma', 0, 0.3),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 0.5, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 0.5, log=True),
                'random_state': 123,
                'eval_metric': 'auc',
                'use_label_encoder': False,
                'n_jobs': -1
            }
            model = XGBClassifier(**params)

        # Crear pipeline con manejo de errores
        try:
            pipeline = Pipeline_imb([
                ('preproc', prep),
                ('sampling', sampler),
                ('classifier', model)
            ])
        except Exception as e:
            # Fallback: usar solo SMOTE
            pipeline = Pipeline_imb([
                ('preproc', prep),
                ('sampling', SMOTE(sampling_strategy=0.7, random_state=123)),
                ('classifier', model)
            ])

        return pipeline

    def objective(trial):
        try:
            pipeline = create_pipeline(trial)

            scores = cross_val_score(
                pipeline,
                X_train,
                y_train,
                cv=cv,
                scoring='roc_auc',
                n_jobs=-1,
                error_score='raise'
            )

            return scores.mean()
        except Exception as e:
            # Si hay error, devolver un valor muy bajo
            return 0.0

    # Crear estudio con manejo de errores
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=123),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
        study_name=f'{model_name}_optimization'
    )

    # Ejecutar optimización con try-except
    try:
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    except Exception as e:
        print(f"Error en optimización: {e}")

    # Verificar si hay trials exitosos
    if len(study.trials) > 0 and study.best_value > 0:
        best_pipeline = create_pipeline(study.best_trial)
        best_pipeline.fit(X_train, y_train)

        return {
            'study': study,
            'best_params': study.best_params,
            'best_value': study.best_value,
            'model': best_pipeline
        }
    else:
        # Modelo por defecto si falla la optimización
        print(f"Usando modelo por defecto para {model_name}")

        if model_name == 'XGBoost':
            model = XGBClassifier(random_state=123, eval_metric='auc', use_label_encoder=False, n_jobs=-1)

        pipeline_default = Pipeline_imb([
            ('preproc', prep),
            ('sampling', SMOTE(sampling_strategy=0.7, random_state=123)),
            ('classifier', model)
        ])

        pipeline_default.fit(X_train, y_train)

        # Calcular valor aproximado
        scores = cross_val_score(pipeline_default, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)

        return {
            'study': None,
            'best_params': {'default': 'default_model'},
            'best_value': scores.mean(),
            'model': pipeline_default
        }

In [9]:
modelos = [
    'XGBoost'
]

resultados_modelos = {}
modelos_entrenados = {}
studies = {}

for modelo in modelos:
    resultados = optimize_model(
        model_name=modelo,
        X_train=X_train,
        y_train=y_train,
        cv=cv,
        n_trials=25  # Reducido para evitar problemas
    )

    resultados_modelos[modelo] = resultados
    modelos_entrenados[modelo] = resultados['model']
    studies[modelo] = resultados['study']

[I 2026-03-22 19:35:44,198] A new study created in memory with name: XGBoost_optimization


Optimizando: XGBoost


  0%|          | 0/25 [00:00<?, ?it/s]

c:\Users\ABUDABI\anaconda3\envs\torch_env\lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [80, 250] and step=40, but the range is not divisible by `step`. It will be replaced with [80, 240].
  optuna_warn(


[I 2026-03-22 19:35:53,341] Trial 0 finished with value: 0.6578583558277522 and parameters: {'sampling_method': 'none', 'sampling_strategy': 0.7102629538165782, 'max_depth': 7, 'n_estimators': 160, 'learning_rate': 0.18880071765466014, 'subsample': 0.905448921575459, 'colsample_bytree': 0.8442795704453083, 'min_child_weight': 3, 'gamma': 0.10295340484526082, 'reg_alpha': 0.004101397183250747, 'reg_lambda': 2.3798772448242304e-05}. Best is trial 0 with value: 0.6578583558277522.


...

c:\Users\ABUDABI\anaconda3\envs\torch_env\lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [80, 250] and step=40, but the range is not divisible by `step`. It will be replaced with [80, 240].
  optuna_warn(


[I 2026-03-22 19:36:19,689] Trial 23 finished with value: 0.6983858572661832 and parameters: {'sampling_method': 'none', 'sampling_strategy': 0.7373468753828109, 'max_depth': 3, 'n_estimators': 240, 'learning_rate': 0.027079011337560407, 'subsample': 0.7701880297718731, 'colsample_bytree': 0.774236064034963, 'min_child_weight': 3, 'gamma': 0.07595086218616087, 'reg_alpha': 0.0007819433521181688, 'reg_lambda': 2.1786176614798712e-08}. Best is trial 15 with value: 0.6991520721131511.


c:\Users\ABUDABI\anaconda3\envs\torch_env\lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [80, 250] and step=40, but the range is not divisible by `step`. It will be replaced with [80, 240].
  optuna_warn(


[I 2026-03-22 19:36:20,246] Trial 24 finished with value: 0.6989077019403745 and parameters: {'sampling_method': 'none', 'sampling_strategy': 0.7832673612893007, 'max_depth': 3, 'n_estimators': 160, 'learning_rate': 0.016980083391608912, 'subsample': 0.7255954635380668, 'colsample_bytree': 0.7254602734872178, 'min_child_weight': 2, 'gamma': 0.12909487105712314, 'reg_alpha': 0.12546546322011637, 'reg_lambda': 1.6258190075047844e-08}. Best is trial 15 with value: 0.6991520721131511.


c:\Users\ABUDABI\anaconda3\envs\torch_env\lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [80, 250] and step=40, but the range is not divisible by `step`. It will be replaced with [80, 240].
  optuna_warn(
c:\Users\ABUDABI\anaconda3\envs\torch_env\lib\site-packages\xgboost\training.py:200: UserWarning: [19:36:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [10]:
# DataFrame para almacenar métricas
metricas_comparativas = []

for nombre_modelo, modelo in modelos_entrenados.items():

    # Predicciones
    y_pred_train = modelo.predict(X_train)
    y_pred_val = modelo.predict(X_val)
    y_pred_final = modelo.predict(X_final)

    y_proba_train = modelo.predict_proba(X_train)[:, 1]
    y_proba_val = modelo.predict_proba(X_val)[:, 1]
    y_proba_final = modelo.predict_proba(X_final)[:, 1]

    # Calcular métricas
    metricas = {
        'Modelo': nombre_modelo,
        'AUC_Train': roc_auc_score(y_train, y_proba_train),
        'AUC_Val': roc_auc_score(y_val, y_proba_val),
        'AUC_Final': roc_auc_score(y_final, y_proba_final),
        'Accuracy_Train': accuracy_score(y_train, y_pred_train),
        'Accuracy_Val': accuracy_score(y_val, y_pred_val),
        'Accuracy_Final': accuracy_score(y_final, y_pred_final),
        'F1_Score_Final': f1_score(y_final, y_pred_final),
        'Precision_Final': precision_score(y_final, y_pred_final),
        'Recall_Final': recall_score(y_final, y_pred_final)
    }

    metricas_comparativas.append(metricas)

# Crear DataFrame comparativo
df_metricas = pd.DataFrame(metricas_comparativas)
df_metricas = df_metricas.sort_values('AUC_Final', ascending=False)


In [11]:
mejor_modelo_idx = df_metricas['AUC_Final'].idxmax()
mejor_modelo_nombre = df_metricas.loc[mejor_modelo_idx, 'Modelo']
mejor_modelo_auc = df_metricas.loc[mejor_modelo_idx, 'AUC_Final']

print(f"\nMEJOR MODELO: {mejor_modelo_nombre}")
print(f"AUC-ROC Final: {mejor_modelo_auc:.4f}")

# Guardar métricas
metricas_path = f'{path_base}/metricas_comparativas.csv'
df_metricas.to_csv(metricas_path, index=False)
print(f"Métricas guardadas")

# Guardar mejor modelo
mejor_modelo_path = f'{path_base}/{mejor_modelo_nombre}.joblib'
joblib.dump(modelos_entrenados[mejor_modelo_nombre], mejor_modelo_path)


MEJOR MODELO: XGBoost
AUC-ROC Final: 0.7018
Métricas guardadas


['C:\\Users\\ABUDABI\\Downloads\\CURSOS_ENEI_2026\\ML_PRODUCCION_WEB\\TRABAJO FINAL\\CLASIFICACION/XGBoost.joblib']